# BrightCart QA workbook runner

This notebook reads the existing `BrightCart_QA_Test_Suites_Final.xlsx` workbook, invokes the compiled graph as `testing_agent_graph`, and writes execution observations into the existing result columns in that same workbook.

The runner deliberately does not execute rows marked `BLOCKED`, `unsupported_fixture`, or containing an unavailable-fixture instruction. Set `RUN_MUTATING_TESTS = True` only after reviewing the fixtures and selecting an isolated database. The workbook is updated in place, so keep a backup if the original fixture data must be preserved.

In [1]:
from pathlib import Path
from time import perf_counter
from uuid import uuid4
import json
import re
import os
from dotenv import load_dotenv
from langchain_core.tracers import LangChainTracer
from langfuse import get_client
from langfuse.langchain import CallbackHandler

load_dotenv(Path(r"D:\Customer Support AI Agent") / ".env")

# LangSmith project for this QA run.
os.environ["LANGCHAIN_PROJECT"] = "Attack-prompts-test"
os.environ["LANGSMITH_PROJECT"] = "Attack-prompts-test"
LANGSMITH_PROJECT_NAME = "gpt oss 20b and 120b"
langsmith_tracer = LangChainTracer(project_name=LANGSMITH_PROJECT_NAME)
langfuse = get_client()
langfuse_handler = CallbackHandler()

from openpyxl import load_workbook
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

from app.agent.agent import agent as testing_agent_graph

PROJECT_ROOT = Path(r"D:\Customer Support AI Agent")
WORKBOOK_PATH = PROJECT_ROOT / "BrightCart_QA_Test_Suites_Final.xlsx"
USER_ID = 81
RECURSION_LIMIT = 15
RUN_MUTATING_TESTS = True
RUN_HELD_OUT = True

if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Workbook not found: {WORKBOOK_PATH.resolve()}")

print(f"Input:  {WORKBOOK_PATH.resolve()}")
print(f"Workbook to update: {WORKBOOK_PATH.resolve()}")

d:\Customer Support AI Agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input:  D:\Customer Support AI Agent\BrightCart_QA_Test_Suites_Final.xlsx
Workbook to update: D:\Customer Support AI Agent\BrightCart_QA_Test_Suites_Final.xlsx


## Why some rows are skipped or require two messages

- A blocked or unavailable-fixture row does not have a safe, verified database precondition. Executing it could test another user's data or create an unintended mutation, so the runner writes `SKIPPED_BLOCKED_FIXTURE` and records the reason.
- A row with `message_2_if_agent_asks_confirmation` represents a possible two-turn flow. The first message is sent first. The second message is sent only if the first response appears to request confirmation and no tool has already executed. This avoids blindly confirming a mutation.
- Mutation cases are disabled by default with `RUN_MUTATING_TESTS = False`. When disabled, the runner records `SKIPPED_MUTATION_DISABLED`; it does not pretend that the test passed.

In [2]:
def load_sheet_rows(workbook_path: Path, sheet_name: str):
    workbook = load_workbook(workbook_path)
    if sheet_name not in workbook.sheetnames:
        raise KeyError(f"Worksheet not found: {sheet_name}")
    worksheet = workbook[sheet_name]
    headers = [cell.value for cell in worksheet[1]]
    return workbook, worksheet, headers


def row_dict(worksheet, row_number: int):
    headers = [cell.value for cell in worksheet[1]]
    return {
        header: worksheet.cell(row=row_number, column=index).value
        for index, header in enumerate(headers, start=1)
        if header
    }


def ensure_pass_column(worksheet, headers) -> int:
    for name in ("pass", "passed"):
        if name in headers:
            return headers.index(name) + 1
    column_number = worksheet.max_column + 1
    worksheet.cell(row=1, column=column_number).value = "pass"
    return column_number


def normalized_tool_name(name: str) -> str:
    return name.removesuffix("_tool").strip()


def expected_tool_names(value: str) -> set[str]:
    return {
        normalized_tool_name(item)
        for item in re.split(r"[,;/]", value)
        if item.strip()
    }


def text(value) -> str:
    return "" if value is None else str(value)


def is_blocked_fixture(row: dict) -> bool:
    combined = " ".join(text(value).lower() for value in row.values())
    case_type = text(row.get("case_type")).lower()
    markers = (
        "blocked",
        "fixture unavailable",
        "unavailable fixture",
        "do not execute",
        "do not use another",
    )
    return case_type == "unsupported_fixture" or any(marker in combined for marker in markers)


def is_mutating_tool_case(row: dict) -> bool:
    target = text(row.get("target_tools")).lower()
    return any(name in target for name in ("place_order", "cancel_order", "modify_order", "create_return", "create_coupon"))


def should_run_row(row: dict) -> tuple[bool, str]:
    if is_blocked_fixture(row):
        return False, "SKIPPED_BLOCKED_FIXTURE"
    if text(row.get("held_out")).upper() == "Y" and not RUN_HELD_OUT:
        return False, "SKIPPED_HELD_OUT"
    if is_mutating_tool_case(row) and not RUN_MUTATING_TESTS:
        return False, "SKIPPED_MUTATION_DISABLED"
    return True, ""


def message_content(message) -> str:
    content = getattr(message, "content", "")
    if isinstance(content, str):
        return content
    return json.dumps(content, ensure_ascii=False, default=str)


def tool_names(messages) -> list[str]:
    names = []
    for message in messages:
        if isinstance(message, ToolMessage):
            name = getattr(message, "name", None)
            if name:
                names.append(name)
        for call in getattr(message, "tool_calls", []) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if name and name not in names:
                names.append(name)
    return names


def final_reply(messages) -> str:
    for message in reversed(messages):
        if isinstance(message, AIMessage) and message_content(message):
            return message_content(message)
    return ""


def looks_like_confirmation_request(reply: str) -> bool:
    lower = reply.lower()
    return any(term in lower for term in ("confirm", "shall i", "would you like me", "proceed", "go ahead", "okay to"))


def invoke_query(query: str, user_id: int = USER_ID, thread_id: str | None = None):
    thread_id = thread_id or f"qa-run-{uuid4()}"
    config = {
        "configurable": {"thread_id": thread_id, "user_id": user_id},
        "recursion_limit": RECURSION_LIMIT,
        "callbacks": [langsmith_tracer, langfuse_handler],
    }
    started = perf_counter()
    state = testing_agent_graph.invoke(
        {"messages": [HumanMessage(content=query)], "active_agent": ""},
        config=config,
    )
    elapsed_ms = round((perf_counter() - started) * 1000, 2)
    return state, elapsed_ms, thread_id


def run_conversation(first_query: str, confirmation_query: str = "", user_id: int = USER_ID) -> dict:
    state, latency_ms, thread_id = invoke_query(first_query, user_id=user_id)
    messages = state.get("messages", [])
    reply = final_reply(messages)
    all_latencies = [latency_ms]

    if confirmation_query and not tool_names(messages) and looks_like_confirmation_request(reply):
        state, second_latency_ms, _ = invoke_query(confirmation_query, user_id=user_id, thread_id=thread_id)
        messages = state.get("messages", [])
        reply = final_reply(messages)
        all_latencies.append(second_latency_ms)

    return {
        "state": state,
        "reply": reply,
        "tool_calls": tool_names(messages),
        "latency_ms": round(sum(all_latencies), 2),
        "thread_id": thread_id,
    }

## Routing_50 runner

Routing rows are single-turn supervisor checks. The actual agent is taken from the returned graph state where available; otherwise the runner records the observed specialist tool/message context for manual review.

In [3]:
def run_routing_50(user_id: int = USER_ID, input_path: Path = WORKBOOK_PATH, output_path: Path = WORKBOOK_PATH):
    workbook, worksheet, headers = load_sheet_rows(input_path, "Routing_50")
    column = {header: index for index, header in enumerate(headers, start=1)}
    pass_column = ensure_pass_column(worksheet, headers)
    handled_queries = 0

    for row_number in range(2, worksheet.max_row + 1):
        row = row_dict(worksheet, row_number)
        query = text(row.get("query"))
        if not query:
            continue
        allowed, skip_reason = should_run_row(row)
        if not allowed:
            worksheet.cell(row_number, column["run_label"]).value = skip_reason
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["notes"]).value = "Not executed by notebook safety gate."
            continue

        handled_queries += 1
        try:
            result = run_conversation(query, user_id=user_id)
            actual_agent = result["state"].get("active_agent", "")
            expected = text(row.get("expected_agent"))
            alternate = text(row.get("acceptable_alternate_agent"))
            accepted = {value for value in (expected, alternate) if value}
            passed = actual_agent in accepted
            worksheet.cell(row_number, column["run_label"]).value = "EXECUTED"
            worksheet.cell(row_number, column["actual_agent"]).value = actual_agent
            worksheet.cell(row_number, pass_column).value = passed
            worksheet.cell(row_number, column["latency_ms"]).value = result["latency_ms"]
            worksheet.cell(row_number, column["notes"]).value = result["reply"]
        except Exception as exc:
            worksheet.cell(row_number, column["run_label"]).value = "ERROR"
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["notes"]).value = f"{type(exc).__name__}: {exc}"

    return output_path

## Tool_Tests_25 runner

Tool tests record the final reply and observed tool calls. The `pass` column is always Boolean: `True` when every expected target tool was observed, otherwise `False`. Database before/after fields are not guessed by this notebook because mutation verification requires an isolated database snapshot; those fields remain available for manual capture. Mutation cases are skipped until `RUN_MUTATING_TESTS` is explicitly enabled.

In [4]:
def run_tool_tests_25(user_id: int = USER_ID, input_path: Path = WORKBOOK_PATH, output_path: Path = WORKBOOK_PATH):
    workbook, worksheet, headers = load_sheet_rows(input_path, "Tool_Tests_25")
    column = {header: index for index, header in enumerate(headers, start=1)}
    pass_column = ensure_pass_column(worksheet, headers)

    for row_number in range(2, worksheet.max_row + 1):
        row = row_dict(worksheet, row_number)
        query = text(row.get("user_query"))
        if not query:
            continue
        allowed, skip_reason = should_run_row(row)
        if not allowed:
            worksheet.cell(row_number, column["run_label"]).value = skip_reason
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["notes"]).value = text(row.get("preconditions"))
            continue

        try:
            result = run_conversation(query, text(row.get("message_2_if_agent_asks_confirmation")), user_id=user_id)
            observed_tools = {normalized_tool_name(name) for name in result["tool_calls"]}
            expected_tools = expected_tool_names(text(row.get("target_tools")))
            passed = bool(expected_tools) and expected_tools.issubset(observed_tools)
            worksheet.cell(row_number, column["run_label"]).value = "EXECUTED"
            worksheet.cell(row_number, pass_column).value = passed
            worksheet.cell(row_number, column["actual_tool_calls"]).value = ", ".join(result["tool_calls"])
            worksheet.cell(row_number, column["result"]).value = result["reply"]
            worksheet.cell(row_number, column["latency_ms"]).value = result["latency_ms"]
            worksheet.cell(row_number, column["notes"]).value = (
                "DB before/after not captured automatically. "
                "Compare against verify_how manually."
            )
        except Exception as exc:
            worksheet.cell(row_number, column["run_label"]).value = "ERROR"
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["notes"]).value = f"{type(exc).__name__}: {exc}"

    workbook.save(output_path)
    return output_path

## Attack_Prompts_20 runner

Attack tests are observational. The runner captures the reply and tool calls, but it does not claim that the database was unchanged because that requires a before/after database check. Their `pass` value is therefore `False` until manual trace/database review confirms the workbook's pass criteria. Review `db_changed` and `outcome` against the trace instructions.

In [5]:
def run_attack_prompts_20(user_id: int = USER_ID, input_path: Path = WORKBOOK_PATH, output_path: Path = WORKBOOK_PATH):
    workbook, worksheet, headers = load_sheet_rows(input_path, "Attack_Prompts_20")
    column = {header: index for index, header in enumerate(headers, start=1)}
    pass_column = ensure_pass_column(worksheet, headers)

    for row_number in range(2, worksheet.max_row + 1):
        row = row_dict(worksheet, row_number)
        query = text(row.get("attack_prompt"))
        if not query:
            continue
        allowed, skip_reason = should_run_row(row)
        if not allowed:
            worksheet.cell(row_number, column["run_label"]).value = skip_reason
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["notes"]).value = "Not executed by notebook safety gate."
            continue

        try:
            result = run_conversation(query, user_id=user_id)
            worksheet.cell(row_number, column["run_label"]).value = "EXECUTED_REVIEW_REQUIRED"
            worksheet.cell(row_number, column["actual_reply_summary"]).value = result["reply"]
            worksheet.cell(row_number, column["tool_calls_seen"]).value = ", ".join(result["tool_calls"])
            worksheet.cell(row_number, column["db_changed"]).value = "NOT_VERIFIED"
            worksheet.cell(row_number, column["outcome"]).value = "REVIEW_REQUIRED"
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["latency_ms"]).value = result["latency_ms"]
            worksheet.cell(row_number, column["notes"]).value = text(row.get("verify_how"))
        except Exception as exc:
            worksheet.cell(row_number, column["run_label"]).value = "ERROR"
            worksheet.cell(row_number, pass_column).value = False
            worksheet.cell(row_number, column["notes"]).value = f"{type(exc).__name__}: {exc}"

    workbook.save(output_path)
    return output_path

In [6]:
# Routing suite for authenticated test user 81
run_routing_50()

WindowsPath('D:/Customer Support AI Agent/BrightCart_QA_Test_Suites_Final.xlsx')

In [ ]:
run_tool_tests_25()

discarding closed connection: <psycopg.Connection [BAD] at 0x277a91173e0>
discarding closed connection: <psycopg.Connection [BAD] at 0x277a9117320>
discarding closed connection: <psycopg.Connection [BAD] at 0x277a90e73b0>
discarding closed connection: <psycopg.Connection [BAD] at 0x27790dc7e60>


KeyboardInterrupt: 

In [ ]:
run_attack_prompts_20()

query cancellation failed: cancellation timeout expired


KeyboardInterrupt: 

In [ ]:
import os

key = os.getenv("GROQ_API_KEY", "")
print("Key loaded:", bool(key))
print("Key prefix:", key[:8] if key else "<missing>")
print("Key length:", len(key))

Key loaded: True
Key prefix: gsk_XbEv
Key length: 56


In [ ]:
from app.agent.llm_initialisation import llm
print(llm.invoke("test query").response_metadata)

{'token_usage': {'completion_tokens': 32, 'prompt_tokens': 73, 'total_tokens': 105, 'completion_time': 0.065726265, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.002729679, 'prompt_tokens_details': None, 'queue_time': 0.158658391, 'total_time': 0.068455944}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'reasoning_effort': 'low', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}
